In [1]:
RUN_TARGET = "colab"  # "colab" | "local"


# Epitope vs Non-Epitope Mutagenesis Sensitivity

This notebook supports both Google Colab and local execution. It tests whether full-sequence single-residue mutagenesis effects are systematically larger inside annotated epitope regions than outside them.

## Setup

### Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1.
2. Use a GPU runtime.
3. Run the notebook from the top. The setup cells mount Drive, bootstrap the shared `xallergen` package, copy or download required assets, and then run the analysis.

### Local
1. Set `RUN_TARGET = "local"` in Cell 1.
2. Run from the repository with the local environment already installed.
3. The Colab setup cells are skipped automatically and outputs are written into the local `results/` tree.

## Outputs

This notebook writes the following artifacts to `results/insilico_mutagenesis/`:
- `full_sequence_saturation_mutagenesis_results.csv`
- `full_sequence_saturation_mutagenesis_annotated.csv`
- `epitope_vs_non_epitope_per_residue.csv`
- `epitope_vs_non_epitope_per_protein.csv`
- `epitope_vs_non_epitope_summary.csv`
- `epitope_vs_non_epitope_comparison.png`
- `epitope_vs_non_epitope_comparison.pdf`
- `main_epitope_vs_non_epitope_mutagenesis.png`
- `main_epitope_vs_non_epitope_mutagenesis.pdf`


In [2]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "pandas": "2.3.2",
        "scipy": "1.16.3",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "seaborn": "0.13.2",
        "matplotlib": "3.10.6",
        "tqdm": "4.67.1",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([_sys.executable, "-m", "pip", "install", "-q", "--upgrade", *_missing_or_mismatched], check=True)
        raise SystemExit("Colab environment updated. Restart the runtime once, then rerun the notebook from the top.")
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Colab environment already compatible. No reinstall needed.


In [3]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    print(f"Google Drive mounted: {DRIVE_ROOT}")
    print(f"Models will sync to: {DRIVE_MODELS}")
    print(f"Results will sync to: {DRIVE_RESULTS}")
else:
    print("Local run: skipping Google Drive mount.")


Mounted at /content/drive
Google Drive mounted: /content/drive/MyDrive/XAllergen2.0
Models will sync to: /content/drive/MyDrive/XAllergen2.0/models
Results will sync to: /content/drive/MyDrive/XAllergen2.0/results


In [4]:
from __future__ import annotations

import csv
import importlib
import os
import shutil as _shutil
import sys
import urllib.request as _urlreq
from pathlib import Path

RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"

if RUN_TARGET == "colab":
    from huggingface_hub import snapshot_download

    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    _SRC_DIR = RUNTIME_ROOT / "src"
    _PACKAGE_DIR = _SRC_DIR / "xallergen"
    _DATA_DIR = RUNTIME_ROOT / "data"
    _MODEL_DIR = RUNTIME_ROOT / "models"
    _RESULTS_DIR = RUNTIME_ROOT / "results"
    _FRESH_ESM2_DIR = RUNTIME_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    for _d in [_PACKAGE_DIR, _DATA_DIR, _MODEL_DIR, _RESULTS_DIR, _FRESH_ESM2_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    for _module_name in [
        "__init__.py",
        "baseline_notebook_utils.py",
        "deep_plant_allergy_utils.py",
        "mtl_epitope_notebook_utils.py",
        "plotting_paper_figures.py",
        "plotting_insilico_mutagenesis.py",
    ]:
        _drive_module = DRIVE_ROOT / "src" / "xallergen" / _module_name
        _target = _PACKAGE_DIR / _module_name
        if _drive_module.exists():
            _shutil.copy2(_drive_module, _target)
            print(f"Copied from Drive src: {_module_name}")
        else:
            _urlreq.urlretrieve(f"{RAW}/src/xallergen/{_module_name}", _target)
            print(f"Downloaded: {_module_name}")

    for _fname in ["positives_splitB.csv"]:
        _drive_path = DRIVE_ROOT / "data" / _fname
        _target = _DATA_DIR / _fname
        if _drive_path.exists():
            _shutil.copy2(_drive_path, _target)
            print(f"Copied data from Drive: {_fname}")
        else:
            _urlreq.urlretrieve(f"{RAW}/data/{_fname}", _target)
            print(f"Downloaded data: {_fname}")

    _checkpoint_name = "baseline_frozen_esm2.pt"
    _drive_checkpoint = DRIVE_MODELS / _checkpoint_name
    _runtime_checkpoint = _MODEL_DIR / _checkpoint_name
    if _drive_checkpoint.exists():
        _shutil.copy2(_drive_checkpoint, _runtime_checkpoint)
        print(f"Copied checkpoint from Drive: {_drive_checkpoint}")
    else:
        _urlreq.urlretrieve(f"{RAW}/models/{_checkpoint_name}", _runtime_checkpoint)
        print(f"Downloaded checkpoint from GitHub: {_checkpoint_name}")

    _drive_hf_dir = DRIVE_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    if _drive_hf_dir.exists():
        os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_drive_hf_dir)
        print(f"Using cached ESM-2 assets from Drive: {_drive_hf_dir}")
    elif _FRESH_ESM2_DIR.exists() and any(_FRESH_ESM2_DIR.iterdir()):
        os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_FRESH_ESM2_DIR)
        print(f"Using cached ESM-2 assets from runtime: {_FRESH_ESM2_DIR}")
    else:
        _cached_path = snapshot_download(
            repo_id="facebook/esm2_t6_8M_UR50D",
            local_dir=_FRESH_ESM2_DIR,
            local_dir_use_symlinks=False,
        )
        os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_cached_path)
        print(f"Fetched ESM-2 assets required by the trained baseline checkpoint: {_cached_path}")

    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    if str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        _src_dir = _candidate / "src"
        if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
            sys.path.insert(0, str(_src_dir))
            RUNTIME_ROOT = _candidate
            break

import matplotlib
if RUN_TARGET == "colab":
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import wilcoxon
from tqdm.auto import tqdm

import xallergen.baseline_notebook_utils as baseline_notebook_utils
importlib.reload(baseline_notebook_utils)

from xallergen.baseline_notebook_utils import (
    annotate_mutagenesis_results,
    build_tokenizer,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    parse_epitope_label,
    print_runtime_context,
    seed_everything,
)

if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    DATA_DIR = RUNTIME_ROOT / "data"
    MODELS_DIR = RUNTIME_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if "DRIVE_RESULTS" in globals() else (RUNTIME_ROOT / "results")
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    configure_matplotlib_cache(Path.cwd())
    PROJECT_ROOT = find_project_root(Path.cwd())
    DATA_DIR = PROJECT_ROOT / "data"
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"
    DEVICE = detect_device()

print_runtime_context(DEVICE, PROJECT_ROOT)
seed_everything(42)
print(f"Run target: {RUN_TARGET}")
print(f"Using project root: {PROJECT_ROOT}")
print(f"Using data dir: {DATA_DIR}")
print(f"Using models dir: {MODELS_DIR}")
print(f"Using results dir: {RESULTS_DIR}")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

# ## Paths and config
ISM_RESULTS_DIR = RESULTS_DIR / "insilico_mutagenesis"
ISM_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TEST_WITH_MASKS_CSV = DATA_DIR / "positives_splitB.csv"
BASELINE_CHECKPOINT_PATH = MODELS_DIR / "baseline_frozen_esm2.pt"
LEGACY_TOPK_MUTAGENESIS_CSV = ISM_RESULTS_DIR / "saturation_mutagenesis_annotated.csv"

FULL_MUTAGENESIS_RESULTS_CSV = ISM_RESULTS_DIR / "full_sequence_saturation_mutagenesis_results.csv"
FULL_MUTAGENESIS_ANNOTATED_CSV = ISM_RESULTS_DIR / "full_sequence_saturation_mutagenesis_annotated.csv"
PER_RESIDUE_CSV = ISM_RESULTS_DIR / "epitope_vs_non_epitope_per_residue.csv"
PER_PROTEIN_CSV = ISM_RESULTS_DIR / "epitope_vs_non_epitope_per_protein.csv"
SUMMARY_CSV = ISM_RESULTS_DIR / "epitope_vs_non_epitope_summary.csv"
FIGURE_PNG = ISM_RESULTS_DIR / "epitope_vs_non_epitope_comparison.png"
FIGURE_PDF = ISM_RESULTS_DIR / "epitope_vs_non_epitope_comparison.pdf"

# Flip REFRESH_FULL_MUTAGENESIS to rebuild the raw and annotated full-sequence mutagenesis caches from scratch.
# Flip REFRESH_REGION_ANALYSIS to rebuild only the downstream summaries and publication figures from cached mutagenesis rows.
REFRESH_FULL_MUTAGENESIS = False
REFRESH_REGION_ANALYSIS = True
MAX_PROTEINS: int | None = None
MODEL_BATCH_SIZE = 128
RESIDUE_CHUNK_SIZE = 16

if not TEST_WITH_MASKS_CSV.exists():
    raise FileNotFoundError(f"Missing positive test split: {TEST_WITH_MASKS_CSV}")
if not BASELINE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing baseline checkpoint: {BASELINE_CHECKPOINT_PATH}")

print(f"Using positives: {TEST_WITH_MASKS_CSV}")
print(f"Using checkpoint: {BASELINE_CHECKPOINT_PATH}")
print(f"Legacy top-k mutagenesis cache exists: {LEGACY_TOPK_MUTAGENESIS_CSV.exists()}")
print(f"Full mutagenesis cache exists: {FULL_MUTAGENESIS_ANNOTATED_CSV.exists()}")


Downloaded: __init__.py
Downloaded: baseline_notebook_utils.py
Downloaded: deep_plant_allergy_utils.py
Downloaded: mtl_epitope_notebook_utils.py
Downloaded: plotting_paper_figures.py
Downloaded: plotting_insilico_mutagenesis.py
Copied data from Drive: positives_splitB.csv
Copied checkpoint from Drive: /content/drive/MyDrive/XAllergen2.0/models/baseline_frozen_esm2.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

.gitattributes: 0.00B [00:00, ?B/s]

Fetched ESM-2 assets required by the trained baseline checkpoint: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

RUN_TARGET: local
Device: cuda
Project root: /content/XAllergen2.0
GPU configuration:
  backend: CUDA
  CUDA available: True
  GPU count: 1
  Current device: NVIDIA A100-SXM4-40GB
Run target: colab
Using project root: /content/XAllergen2.0
Using data dir: /content/XAllergen2.0/data
Using models dir: /content/XAllergen2.0/models
Using results dir: /content/drive/MyDrive/XAllergen2.0/results
Torch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB
Using positives: /content/XAllergen2.0/data/positives_splitB.csv
Using checkpoint: /content/XAllergen2.0/models/baseline_frozen_esm2.pt
Legacy top-k mutagenesis cache exists: True
Full mutagenesis cache exists: True


## Utilities

All helper functions used below are consolidated into the single code cell that follows.


In [5]:
ICML_ONE_COLUMN_WIDTH = 4.2
PAPER_SHORT_HEIGHT = 3.0
PAPER_MEDIUM_HEIGHT = 3.6
PAPER_DPI = 300
EPITOPE_COLOR = "#C44E52"
NON_EPITOPE_COLOR = "#4C72B0"
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")


def set_paper_plot_style() -> None:
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 11,
        "axes.titlesize": 11,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "legend.fontsize": 9,
        "legend.frameon": False,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def significance_marker(p_value: float) -> str:
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


def bootstrap_mean_ci(values, n_bootstrap: int = 5000, random_state: int = 42) -> tuple[float, float, float]:
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    mean_value = float(values.mean())
    if values.size == 1:
        return mean_value, mean_value, mean_value
    rng = np.random.default_rng(random_state)
    boot = np.empty(n_bootstrap, dtype=float)
    for idx in range(n_bootstrap):
        sample = rng.choice(values, size=values.size, replace=True)
        boot[idx] = sample.mean()
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
    return mean_value, float(ci_low), float(ci_high)


def load_positive_test_sequences(csv_path: Path, max_proteins: int | None = None) -> pd.DataFrame:
    df = pd.read_csv(csv_path).copy()
    df["sequence_id"] = df.get("sequence_id", df["accession"]).astype(str)
    df = df.drop_duplicates(subset=["sequence_id"]).reset_index(drop=True)
    if max_proteins is not None:
        df = df.head(int(max_proteins)).copy()
    df["sequence"] = df["sequence"].astype(str)
    df["seq_len"] = df["sequence"].str.len().astype(int)
    df["epitope_label"] = [
        parse_epitope_label(seq, start, end)
        for seq, start, end in zip(df["sequence"], df["epitope_start"], df["epitope_end"])
    ]
    df["n_epitope_residues"] = df["epitope_label"].map(lambda arr: int(np.asarray(arr).sum()))
    df["n_non_epitope_residues"] = df["seq_len"] - df["n_epitope_residues"]
    return df


def _forward_probabilities(model, tokenizer, sequences: list[str], device: str, batch_size: int = 128) -> list[float]:
    probs: list[float] = []
    model.eval()
    for start in range(0, len(sequences), batch_size):
        batch_sequences = sequences[start : start + batch_size]
        enc = tokenizer(
            batch_sequences,
            add_special_tokens=False,
            padding=True,
            truncation=False,
            return_tensors="pt",
        )
        enc = {key: value.to(device) for key, value in enc.items()}
        with torch.no_grad():
            outputs = model(enc["input_ids"], enc["attention_mask"])
        batch_probs = torch.sigmoid(outputs["logits"]).detach().cpu().numpy().astype(float)
        probs.extend(batch_probs.tolist())
    return probs


def run_batched_full_sequence_mutagenesis(
    model,
    tokenizer,
    sequence: str,
    device: str,
    model_batch_size: int = 128,
    residue_chunk_size: int = 16,
) -> pd.DataFrame:
    residues = list(sequence)
    p_base = _forward_probabilities(model, tokenizer, [sequence], device=device, batch_size=1)[0]
    rows: list[dict[str, object]] = []
    for start_idx in range(0, len(residues), residue_chunk_size):
        chunk_indices = list(range(start_idx, min(start_idx + residue_chunk_size, len(residues))))
        mutated_sequences: list[str] = []
        metadata: list[tuple[int, str, str]] = []
        for residue_idx in chunk_indices:
            original_aa = residues[residue_idx]
            for mutant_aa in AMINO_ACIDS:
                if mutant_aa == original_aa:
                    continue
                mutated = residues.copy()
                mutated[residue_idx] = mutant_aa
                mutated_sequences.append("".join(mutated))
                metadata.append((residue_idx, original_aa, mutant_aa))
        mutant_probs = _forward_probabilities(
            model,
            tokenizer,
            mutated_sequences,
            device=device,
            batch_size=model_batch_size,
        )
        for (residue_idx, original_aa, mutant_aa), p_mutant in zip(metadata, mutant_probs):
            delta_p = float(p_base) - float(p_mutant)
            rows.append(
                {
                    "position": int(residue_idx),
                    "original_aa": original_aa,
                    "mutant_aa": mutant_aa,
                    "p_base": float(p_base),
                    "p_mutant": float(p_mutant),
                    "delta_p": float(delta_p),
                    "reduces_allergenicity": bool(delta_p > 0),
                }
            )
    return pd.DataFrame(rows)


def _completed_sequence_ids_from_raw_csv(path: Path) -> set[str]:
    if not path.exists():
        return set()
    try:
        df = pd.read_csv(path, usecols=["sequence_id"])
    except Exception:
        return set()
    return set(df["sequence_id"].dropna().astype(str).unique())


def annotate_epitope_membership(mut_df: pd.DataFrame, positive_df: pd.DataFrame) -> pd.DataFrame:
    lookup = positive_df.set_index("sequence_id")[["sequence", "epitope_label", "seq_len"]].to_dict(orient="index")
    annotated_parts: list[pd.DataFrame] = []
    for sequence_id, group_df in mut_df.groupby("sequence_id", sort=False):
        key = str(sequence_id)
        if key not in lookup:
            raise ValueError(f"Mutagenesis rows contain unknown sequence_id: {key}")
        seq_info = lookup[key]
        epitope_label = np.asarray(seq_info["epitope_label"], dtype=np.float32)
        seq_len = int(seq_info["seq_len"])
        max_position = int(group_df["position"].max()) if not group_df.empty else -1
        if max_position >= seq_len:
            raise ValueError(
                f"sequence_id={key} has mutagenesis position {max_position} outside sequence length {seq_len}"
            )
        part = group_df.copy()
        part["position"] = part["position"].astype(int)
        part["residue_index_1based"] = part["position"] + 1
        part["is_epitope_residue"] = part["position"].map(lambda idx: bool(epitope_label[int(idx)]))
        part["abs_delta_p"] = part["delta_p"].abs().astype(float)
        annotated_parts.append(part)
    return pd.concat(annotated_parts, ignore_index=True)


def summarize_region_effects(annotated_mut_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    residue_df = (
        annotated_mut_df
        .groupby(["sequence_id", "position", "residue_index_1based", "original_aa", "is_epitope_residue"], as_index=False)
        .agg(
            n_mutations=("mutant_aa", "count"),
            mean_delta_p=("delta_p", "mean"),
            mean_abs_delta_p=("abs_delta_p", "mean"),
            frac_reducing=("reduces_allergenicity", lambda values: float(pd.Series(values).astype(float).mean())),
        )
    )
    protein_region_df = (
        residue_df
        .groupby(["sequence_id", "is_epitope_residue"], as_index=False)
        .agg(
            n_residues=("position", "nunique"),
            mean_delta_p=("mean_delta_p", "mean"),
            mean_abs_delta_p=("mean_abs_delta_p", "mean"),
            frac_reducing=("frac_reducing", "mean"),
        )
    )
    wide = protein_region_df.pivot(index="sequence_id", columns="is_epitope_residue")
    wide.columns = [f"{metric}_{'epitope' if is_epi else 'non_epitope'}" for metric, is_epi in wide.columns]
    wide = wide.dropna().reset_index()
    for metric in ["mean_delta_p", "mean_abs_delta_p", "frac_reducing"]:
        wide[f"{metric}_diff_epitope_minus_non_epitope"] = (
            wide[f"{metric}_epitope"] - wide[f"{metric}_non_epitope"]
        )
    return residue_df, wide


def paired_metric_summary(per_protein_df: pd.DataFrame, metric: str, label: str) -> dict[str, float | int | str]:
    epitope = per_protein_df[f"{metric}_epitope"].to_numpy(dtype=float)
    non_epitope = per_protein_df[f"{metric}_non_epitope"].to_numpy(dtype=float)
    diff = epitope - non_epitope
    mean_diff, ci_low, ci_high = bootstrap_mean_ci(diff)
    return {
        "metric": metric,
        "label": label,
        "n_proteins": int(len(diff)),
        "mean_epitope": float(np.mean(epitope)),
        "mean_non_epitope": float(np.mean(non_epitope)),
        "median_epitope": float(np.median(epitope)),
        "median_non_epitope": float(np.median(non_epitope)),
        "mean_diff_epitope_minus_non_epitope": float(mean_diff),
        "mean_diff_ci95_low": float(ci_low),
        "mean_diff_ci95_high": float(ci_high),
        "proteins_epitope_gt_non_epitope": int((diff > 0).sum()),
        "proteins_non_epitope_gt_epitope": int((diff < 0).sum()),
        "proteins_equal": int((diff == 0).sum()),
        "wilcoxon_p_two_sided": float(wilcoxon(epitope, non_epitope, alternative="two-sided").pvalue),
        "wilcoxon_p_epitope_greater": float(wilcoxon(epitope, non_epitope, alternative="greater").pvalue),
        "wilcoxon_p_epitope_less": float(wilcoxon(epitope, non_epitope, alternative="less").pvalue),
    }


def plot_paired_region_comparison(per_protein_df: pd.DataFrame, summary_df: pd.DataFrame, png_path: Path, pdf_path: Path) -> None:
    metric_specs = [
        ("mean_abs_delta_p", "Mean |Δp|", "Primary endpoint"),
        ("mean_delta_p", "Mean Δp", "Directional endpoint"),
        ("frac_reducing", "Fraction reducing", "Reduction frequency"),
    ]
    set_paper_plot_style()
    fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.8))
    for ax, (metric, ylabel, title) in zip(axes, metric_specs):
        x_positions = [0, 1]
        epi = per_protein_df[f"{metric}_epitope"].to_numpy(dtype=float)
        non = per_protein_df[f"{metric}_non_epitope"].to_numpy(dtype=float)
        for non_value, epi_value in zip(non, epi):
            ax.plot(x_positions, [non_value, epi_value], color="lightgray", alpha=0.8, linewidth=1.0, zorder=1)
        ax.scatter(np.zeros_like(non), non, color=NON_EPITOPE_COLOR, s=20, zorder=3, label="Non-epitope")
        ax.scatter(np.ones_like(epi), epi, color=EPITOPE_COLOR, s=20, zorder=3, label="Epitope")
        non_mean, _, _ = bootstrap_mean_ci(non)
        epi_mean, _, _ = bootstrap_mean_ci(epi)
        ax.scatter(0, non_mean, color="black", s=90, marker="_", linewidths=2.5, zorder=4)
        ax.scatter(1, epi_mean, color="black", s=90, marker="_", linewidths=2.5, zorder=4)
        row = summary_df.loc[summary_df["metric"] == metric].iloc[0]
        ax.set_xticks(x_positions)
        ax.set_xticklabels(["Non-epitope", "Epitope"], rotation=20)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.text(
            0.5,
            0.98,
            f"paired diff = {row['mean_diff_epitope_minus_non_epitope']:.4f}\n"
            f"95% CI [{row['mean_diff_ci95_low']:.4f}, {row['mean_diff_ci95_high']:.4f}]\n"
            f"Wilcoxon p(epi > non) = {row['wilcoxon_p_epitope_greater']:.3g}\n"
            f"Wilcoxon p(two-sided) = {row['wilcoxon_p_two_sided']:.3g}",
            ha="center",
            va="top",
            transform=ax.transAxes,
            fontsize=9,
            bbox={"facecolor": "white", "edgecolor": "#d9d9d9", "boxstyle": "round,pad=0.3", "alpha": 0.9},
        )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=NON_EPITOPE_COLOR, markersize=6.5, label="Non-epitope"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=EPITOPE_COLOR, markersize=6.5, label="Epitope"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0.0, 0.04, 1.0, 1.0))
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


## Load splitB positives

This analysis uses all proteins in `positives_splitB.csv` with merged epitope intervals converted into dense residue masks via `parse_epitope_label(...)`.

This is intentionally broader than the IG-masking subset: the epitope-versus-non-epitope mutagenesis analysis uses all 61 splitB positives rather than the smaller IG-validated subset.


In [6]:
positive_test_df = load_positive_test_sequences(TEST_WITH_MASKS_CSV, max_proteins=MAX_PROTEINS)
total_residues = int(positive_test_df["seq_len"].sum())
total_single_mutants = int((positive_test_df["seq_len"] * 19).sum())
print(f"Positive test proteins loaded: {len(positive_test_df)}")
print(f"Total residues across proteins: {total_residues}")
print(f"Total single mutants to evaluate: {total_single_mutants}")
print(f"Mean sequence length: {positive_test_df['seq_len'].mean():.1f}")
print(f"Mean epitope coverage: {positive_test_df['epitope_coverage'].astype(float).mean():.3f}")


Positive test proteins loaded: 61
Total residues across proteins: 17133
Total single mutants to evaluate: 325527
Mean sequence length: 280.9
Mean epitope coverage: 0.250


## Build or Resume the Full-Sequence Mutagenesis Cache

This cell is the main GPU workload. It reuses `full_sequence_saturation_mutagenesis_annotated.csv` when available unless `REFRESH_FULL_MUTAGENESIS` is set to `True`.

Progress is appended incrementally to `full_sequence_saturation_mutagenesis_results.csv`, so interrupted runs can resume without recomputing completed proteins.


In [7]:
if FULL_MUTAGENESIS_ANNOTATED_CSV.exists() and not REFRESH_FULL_MUTAGENESIS:
    full_mutagenesis_annotated_df = pd.read_csv(FULL_MUTAGENESIS_ANNOTATED_CSV)
    print(f"Loaded existing full-sequence cache: {FULL_MUTAGENESIS_ANNOTATED_CSV}")
else:
    model, _ = load_baseline_checkpoint(BASELINE_CHECKPOINT_PATH, DEVICE)
    tokenizer = build_tokenizer()
    completed_sequence_ids = set() if REFRESH_FULL_MUTAGENESIS else _completed_sequence_ids_from_raw_csv(FULL_MUTAGENESIS_RESULTS_CSV)
    print(f"Already completed proteins in raw cache: {len(completed_sequence_ids)}")
    proteins_to_run = positive_test_df.loc[~positive_test_df['sequence_id'].astype(str).isin(completed_sequence_ids)].copy()
    print(f"Proteins remaining to run: {len(proteins_to_run)}")

    if REFRESH_FULL_MUTAGENESIS and FULL_MUTAGENESIS_RESULTS_CSV.exists():
        FULL_MUTAGENESIS_RESULTS_CSV.unlink()

    for row in tqdm(
        proteins_to_run.itertuples(index=False),
        total=len(proteins_to_run),
        desc="Full-sequence saturation mutagenesis",
        unit="protein",
    ):
        mut_df = run_batched_full_sequence_mutagenesis(
            model=model,
            tokenizer=tokenizer,
            sequence=row.sequence,
            device=DEVICE,
            model_batch_size=MODEL_BATCH_SIZE,
            residue_chunk_size=RESIDUE_CHUNK_SIZE,
        )
        mut_df["sequence_id"] = str(row.sequence_id)
        mut_df.to_csv(
            FULL_MUTAGENESIS_RESULTS_CSV,
            mode="a",
            index=False,
            header=not FULL_MUTAGENESIS_RESULTS_CSV.exists(),
        )

    full_mutagenesis_df = pd.read_csv(FULL_MUTAGENESIS_RESULTS_CSV)
    full_mutagenesis_annotated_df = annotate_mutagenesis_results(full_mutagenesis_df)
    full_mutagenesis_annotated_df.to_csv(FULL_MUTAGENESIS_ANNOTATED_CSV, index=False)
    print(f"Saved raw full-sequence cache to: {FULL_MUTAGENESIS_RESULTS_CSV}")
    print(f"Saved annotated full-sequence cache to: {FULL_MUTAGENESIS_ANNOTATED_CSV}")

print(f"Mutagenesis proteins: {full_mutagenesis_annotated_df['sequence_id'].nunique()}")
print(f"Mutagenesis rows: {len(full_mutagenesis_annotated_df):,}")

Loaded existing full-sequence cache: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/full_sequence_saturation_mutagenesis_annotated.csv
Mutagenesis proteins: 61
Mutagenesis rows: 325,527


## Region Comparison Analysis

Each mutation row is mapped to epitope versus non-epitope using the curated residue mask. The notebook then writes the paired per-residue and per-protein CSVs, the summary CSV, and the two retained publication figures.


In [8]:
if SUMMARY_CSV.exists() and PER_PROTEIN_CSV.exists() and PER_RESIDUE_CSV.exists() and not REFRESH_REGION_ANALYSIS:
    per_residue_df = pd.read_csv(PER_RESIDUE_CSV)
    per_protein_df = pd.read_csv(PER_PROTEIN_CSV)
    summary_df = pd.read_csv(SUMMARY_CSV)
    print("Loaded existing region-comparison outputs.")
else:
    region_annotated_df = annotate_epitope_membership(full_mutagenesis_annotated_df, positive_test_df)
    per_residue_df, per_protein_df = summarize_region_effects(region_annotated_df)
    summary_rows = [
        paired_metric_summary(per_protein_df, "mean_abs_delta_p", "Primary: mean absolute probability shift"),
        paired_metric_summary(per_protein_df, "mean_delta_p", "Secondary: signed probability shift"),
        paired_metric_summary(per_protein_df, "frac_reducing", "Secondary: fraction of reducing substitutions"),
    ]
    summary_df = pd.DataFrame(summary_rows)
    per_residue_df.to_csv(PER_RESIDUE_CSV, index=False)
    per_protein_df.to_csv(PER_PROTEIN_CSV, index=False)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    plot_paired_region_comparison(per_protein_df, summary_df, FIGURE_PNG, FIGURE_PDF)
    print(f"Saved per-residue summary to: {PER_RESIDUE_CSV}")
    print(f"Saved per-protein summary to: {PER_PROTEIN_CSV}")
    print(f"Saved summary table to: {SUMMARY_CSV}")
    print(f"Saved figure: {FIGURE_PNG}")
    print(f"Saved figure: {FIGURE_PDF}")

summary_df

Saved per-residue summary to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/epitope_vs_non_epitope_per_residue.csv
Saved per-protein summary to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/epitope_vs_non_epitope_per_protein.csv
Saved summary table to: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/epitope_vs_non_epitope_summary.csv
Saved figure: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/epitope_vs_non_epitope_comparison.png
Saved figure: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/epitope_vs_non_epitope_comparison.pdf


,metric,label,n_proteins,mean_epitope,mean_non_epitope,median_epitope,median_non_epitope,mean_diff_epitope_minus_non_epitope,mean_diff_ci95_low,mean_diff_ci95_high,proteins_epitope_gt_non_epitope,proteins_non_epitope_gt_epitope,proteins_equal,wilcoxon_p_two_sided,wilcoxon_p_epitope_greater,wilcoxon_p_epitope_less
0,mean_abs_delta_p,Primary: mean absolute probability shift,61,0.012464,0.013322,0.007166,0.007563,-0.000858,-0.002725,0.001175,24,37,0,0.020534,0.989733,0.010267
1,mean_delta_p,Secondary: signed probability shift,61,0.001380,0.001418,0.000464,0.000547,-0.000037,-0.001365,0.001391,32,29,0,0.945597,0.527201,0.472799
2,frac_reducing,Secondary: fraction of reducing substitutions,61,0.532377,0.530842,0.535955,0.518668,0.001535,-0.022016,0.025462,28,33,0,0.871611,0.435806,0.564194


## Interpretation guide

- `mean_abs_delta_p` is the primary endpoint and anchors the main paper figure and inferential claim.
- Larger `mean_abs_delta_p` inside epitopes supports region-specific sensitivity, whereas similar values argue against epitope-focused localization.
- Larger effects outside epitopes are more consistent with broader sequence-context reliance than with direct epitope recovery.


In [9]:
MAIN_FIGURE_PNG = RESULTS_DIR / "insilico_mutagenesis" / "main_epitope_vs_non_epitope_mutagenesis.png"
MAIN_FIGURE_PDF = RESULTS_DIR / "insilico_mutagenesis" / "main_epitope_vs_non_epitope_mutagenesis.pdf"

set_paper_plot_style()

metric = "mean_abs_delta_p"
epitope_values = per_protein_df[f"{metric}_epitope"].to_numpy(dtype=float)
non_epitope_values = per_protein_df[f"{metric}_non_epitope"].to_numpy(dtype=float)
p_two_sided = float(summary_df.loc[summary_df["metric"].eq(metric), "wilcoxon_p_two_sided"].iloc[0])

rng = np.random.default_rng(42)
x_positions = np.array([0.0, 1.0])
non_epitope_jitter = rng.uniform(-0.06, 0.06, size=len(non_epitope_values))
epitope_jitter = rng.uniform(-0.06, 0.06, size=len(epitope_values))

fig, ax = plt.subplots(figsize=(3.5, 3.8))
violin = ax.violinplot(
    [non_epitope_values, epitope_values],
    positions=x_positions,
    widths=0.58,
    showmeans=False,
    showmedians=False,
    showextrema=False,
)
for body, color in zip(violin["bodies"], [NON_EPITOPE_COLOR, EPITOPE_COLOR]):
    body.set_facecolor(color)
    body.set_edgecolor(color)
    body.set_alpha(0.26)
    body.set_linewidth(1.0)

for non_value, epi_value in zip(non_epitope_values, epitope_values):
    ax.plot(x_positions, [non_value, epi_value], color="lightgray", linewidth=1.0, alpha=0.85, zorder=1)

ax.scatter(np.full(len(non_epitope_values), x_positions[0]) + non_epitope_jitter, non_epitope_values, s=18, color=NON_EPITOPE_COLOR, alpha=0.90, zorder=3)
ax.scatter(np.full(len(epitope_values), x_positions[1]) + epitope_jitter, epitope_values, s=18, color=EPITOPE_COLOR, alpha=0.90, zorder=3)

for xpos, values in zip(x_positions, [non_epitope_values, epitope_values]):
    mean_value, ci_low, ci_high = bootstrap_mean_ci(values)
    ax.errorbar(
        [xpos],
        [mean_value],
        yerr=[[mean_value - ci_low], [ci_high - mean_value]],
        fmt="o",
        color="black",
        markerfacecolor="black",
        markeredgecolor="black",
        markersize=4.8,
        elinewidth=1.3,
        capsize=3,
        zorder=5,
    )

y_data_max = float(np.nanmax(np.concatenate([non_epitope_values, epitope_values])))
_, non_ci_low, non_ci_high = bootstrap_mean_ci(non_epitope_values)
_, epi_ci_low, epi_ci_high = bootstrap_mean_ci(epitope_values)
y_ref_max = max(y_data_max, float(non_ci_high), float(epi_ci_high))
y_ref_min = float(np.nanmin(np.concatenate([non_epitope_values, epitope_values, [non_ci_low, epi_ci_low]])))
y_range = max(y_ref_max - y_ref_min, 1e-6)
bracket_y = y_ref_max + 0.08 * y_range
bracket_h = 0.035 * y_range
ax.plot(
    [x_positions[0], x_positions[0], x_positions[1], x_positions[1]],
    [bracket_y, bracket_y + bracket_h, bracket_y + bracket_h, bracket_y],
    color="black",
    linewidth=1.1,
    clip_on=False,
)
ax.text(0.5, bracket_y + bracket_h + 0.012 * y_range, significance_marker(p_two_sided), ha="center", va="bottom", fontsize=9)
ax.text(
    0.5,
    bracket_y + bracket_h + 0.050 * y_range,
    f"non-epi > epi, Wilcoxon p = {p_two_sided:.2e} (two-sided)",
    ha="center",
    va="bottom",
    fontsize=9,
)

ax.set_xticks(x_positions)
ax.set_xticklabels(["Non-epitope", "Epitope"])
for tick_label, color in zip(ax.get_xticklabels(), [NON_EPITOPE_COLOR, EPITOPE_COLOR]):
    tick_label.set_color(color)
ax.set_ylabel("Mean |Δp| per residue")
ax.set_xlim(-0.45, 1.45)
ax.set_ylim(y_ref_min - 0.05 * y_range, bracket_y + bracket_h + 0.14 * y_range)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(MAIN_FIGURE_PDF, dpi=300, bbox_inches="tight")
fig.savefig(MAIN_FIGURE_PNG, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved paper figure: {MAIN_FIGURE_PNG}")
print(f"Saved paper figure: {MAIN_FIGURE_PDF}")


Saved paper figure: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/main_epitope_vs_non_epitope_mutagenesis.png
Saved paper figure: /content/drive/MyDrive/XAllergen2.0/results/insilico_mutagenesis/main_epitope_vs_non_epitope_mutagenesis.pdf
